# The risk budget: Euler contributions against what was declared

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.budget.euler`

**Modules covered** `budget/euler.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The budget layer decomposes each cell's risk by Euler contributions and reads the result against the vector the mandate declared before construction: 55% equity, 20% government, 15% credit, 10% real and cash. The entry point prints the realised shares beside the target, the additivity residual, and the ex-ante against ex-post tracking error.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `budget/euler.py::additivity` traces to the additivity condition of a measure homogeneous of degree one, checked numerically on every run rather than assumed
- `budget/euler.py::budget` traces to Euler's theorem applied to a risk measure homogeneous of degree one: the component contributions risk systems report as the risk budget
- `budget/euler.py::contributions` traces to Euler component contributions: `w_i d(sigma)/d(w_i)` in percentage of volatility
- `budget/euler.py::diversification` traces to Euler's theorem applied to a risk measure homogeneous of degree one: the component contributions risk systems report as the risk budget
- `budget/euler.py::ex_ante_tracking_error` traces to the ex-ante tracking error the risk model predicts, against the ex-post one the realised series measures - the forecast-quality pair of this effort's risk-budgeting design
- `budget/euler.py::ex_post_tracking_error` traces to Euler's theorem applied to a risk measure homogeneous of degree one: the component contributions risk systems report as the risk budget
- `budget/euler.py::forecast_quality` traces to Euler's theorem applied to a risk measure homogeneous of degree one: the component contributions risk systems report as the risk budget
- `budget/euler.py::main` traces to Euler's theorem applied to a risk measure homogeneous of degree one: the component contributions risk systems report as the risk budget
- `budget/euler.py::path_contributions` traces to component contributions along the realised path of a book rather than at one date, which is what the risk budget is read against
- `budget/euler.py::report` traces to Euler's theorem applied to a risk measure homogeneous of degree one: the component contributions risk systems report as the risk budget
- `budget/euler.py::tail_contributions` traces to the tail decomposition of expected shortfall, which does satisfy the additivity the value at risk does not
- `budget/euler.py::var_refusal` traces to the coherence failure of value at risk (Artzner, Delbaen, Eber & Heath 1999, Mathematical Finance 9(3)): its conditional contributions do not sum to it, which is measured here rather than asserted

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`budget/euler.py`**

The Euler risk budget, the condition it adds up under, and the refusal that condition forces.

**Components sum to total risk when the risk measure is homogeneous of degree one and the
decomposition is the Euler one.** That is the condition, stated rather than assumed, and it is why the
budget is denominated in **percentage contribution to volatility**: volatility is homogeneous of degree
one and its gradient decomposition is exact, so the shares decompose the total rather than merely
resembling its parts. Expected shortfall satisfies the same condition.

**Value at risk does not, so VaR contributions are refused.** The refusal is verified rather than
asserted: `var_refusal` computes the conditional contributions a VaR report normally prints and shows
what they actually sum to. They sum to the expected shortfall, not to the value at risk - one is the
mean of the tail and the other is its edge - so a report presenting them as VaR contributions
overstates the quantity it claims to decompose. The failing case is kept as a documented negative, and
the positive case falls out of the same computation.

**Marginal contributions are reported and incremental ones are not.** The marginal contribution is the
gradient `d(sigma)/d(w_i)`, a property of the portfolio; an incremental contribution is the change in
risk from adding a sleeve, which depends on where the sleeve is added and in what order the others
arrived. Reporting an order-dependent number beside an order-independent one invites a reader to read
the first as the second, so the second appears only in the hand-checked case.

**The budget is read against the vector the mandate declared, on the out-of-sample window's own
covariance.** The covariance is held fixed across the run rather than re-estimated monthly, because the
question the budget answers is where the risk *went*: a moving covariance would make the consumption
and the declared target two moving objects that never meet. What moves is the book, which is the thing
under test. The declared vector is fixed in the data layer before construction, which is what makes
"consumed by design or by accident" a measurable quantity rather than a claim.

**Ex-ante against ex-post tracking error, with the difference as its own result.** The ex-ante number
comes from the covariance and the book; the ex-post number is the realised dispersion of the active
return. Their difference is a forecast-quality metric and is reported as its own line rather than left
for a reader to subtract.

## 3. The data contract it consumes, and the as-of rule

Euler contributions in percentage of volatility, weighted by the covariance the cell was built on, aggregated into the declared budget groups in the data layer's own order. Additivity is checked numerically on every run rather than assumed, and value at risk is refused as a measure to decompose: its conditional contributions do not sum to it, which is measured rather than asserted.

## 4. The worked example on small numbers, with the identity checked

For two sleeves with a diagonal covariance the contributions are hand-checkable: sleeve a's share is its own variance over the portfolio's, and the two sum to the portfolio volatility.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import numpy as np

from portfolio_workbench.budget import euler

weights = np.array([0.6, 0.4])
covariance = np.array([[0.04, 0.0], [0.0, 0.01]])
table = euler.contributions(weights, covariance)

volatility = float(np.sqrt(weights @ covariance @ weights))
assert abs(table["contribution"].sum() - volatility) < 1e-15
assert abs(table["share"].sum() - 1.0) < 1e-15
assert abs(table.loc[0, "share"] - 0.36 * 0.04 / volatility ** 2) < 1e-15
print(f"volatility {volatility:.6f}, shares {table['share'].round(4).tolist()}")

volatility 0.126491, shares [0.9, 0.1]


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.budget import euler
from portfolio_workbench.data import universe

print(f"declared budget: {universe.RISK_BUDGET}")
print(f"additivity tolerance {euler.ADDITIVITY_TOLERANCE:.0e} relative, checked on every run")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
declared budget: {'equity': 0.55, 'government': 0.2, 'credit': 0.15, 'real_and_cash': 0.1}
additivity tolerance 1e-10 relative, checked on every run


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.budget.euler"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[risk] snapshot 2026-09-13: the budget is read over 131 out-of-sample months 2015-09..2026-07, one covariance held across the run
[risk] the declared vector is fixed in the data layer before construction: equity 55%, government 20%, credit 15%, real_and_cash 10% of total volatility, which is the denomination the mandate itself states
[risk] components sum to total risk when the measure is homogeneous of degree one and the decomposition is the Euler one - true for volatility and expected shortfall, false for the value at risk: checked numerically to 1e-10 relative on every run
[risk] run                                  equity  government   credit  real_and_cash   vol/yr  div ratio  additivity
[risk] equal_weight                          48.3%       16.4%    13.1%          22.2%    6.59%       1.38    3.47e-18
[risk] mean_variance_shrunk                  75.8%        4.9%     0.1%          19.2%   10.63%       1.35    0.00e+00
[risk] minimum_variance                       3.1%        7.

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The gap between realised and target is the answer to whether the budget was consumed by design or by accident, and it is a property of the constructor: a risk-based family concentrates volatility where the covariance says the risk is, which may not be where the budget says it should be. The resolution limit is the window's own covariance, so a gap measured on one estimation window moves with the next. A reader must not read a realised share near its target as evidence that the constructor targeted it, since a book can land near a target for reasons the objective never mentioned.

## 7. What this module does not establish

Nothing here establishes that the declared vector is the right budget for the mandate, or that a realised share far from it is a failure: it is a description of the book the objective produced. Nothing here decomposes tail risk additively beyond expected shortfall, and the value-at-risk refusal is a statement about additivity rather than about the measure's usefulness in other roles.